In [1]:
# importações, caminhos, categorias e funções auxiliares
import os
from pathlib import Path

import kaleido
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

KALEIDO_CHROME_PATH = Path(kaleido.get_chrome_sync())
if not KALEIDO_CHROME_PATH.is_file():
    raise FileNotFoundError(
        f"Chrome gerenciado pelo Kaleido não encontrado: {KALEIDO_CHROME_PATH}"
    )
os.environ["BROWSER_PATH"] = str(KALEIDO_CHROME_PATH)
print(f"Chrome usado pelo Kaleido: {KALEIDO_CHROME_PATH}")

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
TRAINING_RESULTS_DIR = ROOT / "results" / "model_training" / "randomized_search"
EVALUATION_RESULTS_DIR = ROOT / "results" / "model_evaluation"
ANALYSIS_DIR = ROOT / "results" / "results_analysis"
TABLES_DIR = ANALYSIS_DIR / "tables"
FIGURES_DIR = ANALYSIS_DIR / "figures"
SVG_DIR = FIGURES_DIR / "svg"
PNG_DIR = FIGURES_DIR / "png"

for output_directory in [TABLES_DIR, SVG_DIR, PNG_DIR]:
    output_directory.mkdir(parents=True, exist_ok=True)

for output_directory, output_pattern in [
    (TABLES_DIR, "*.csv"),
    (SVG_DIR, "*.svg"),
    (PNG_DIR, "*.png"),
]:
    for previous_output in output_directory.glob(output_pattern):
        previous_output.unlink()

MODEL_ORDER = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
SCENARIO_ORDER = ["intra_dataset", "cross_dataset", "combined_to_unseen"]
TRAINING_SOURCE_ORDER = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
    "cicids2017_unsw_nb15",
    "cicids2017_iot23",
    "unsw_nb15_iot23",
]
TEST_SOURCE_ORDER = ["cicids2017", "unsw_nb15", "iot23"]

MODEL_LABELS = {
    "decision_tree": "Árvore de decisão",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "linear_svm": "SVM linear",
    "mlp": "MLP",
}
MODEL_ABBREVIATIONS = {
    "decision_tree": "DT",
    "random_forest": "RF",
    "xgboost": "XGB",
    "linear_svm": "SVM",
    "mlp": "MLP",
}
SCENARIO_LABELS = {
    "intra_dataset": "Intra-dataset",
    "cross_dataset": "Cross-dataset",
    "combined_to_unseen": "Combinado",
}
SCENARIO_FILE_LABELS = {
    "intra_dataset": "intra_dataset",
    "cross_dataset": "cross_dataset",
    "combined_to_unseen": "combined",
}
SOURCE_LABELS = {
    "cicids2017": "CICIDS2017",
    "unsw_nb15": "UNSW-NB15",
    "iot23": "IoT-23",
    "cicids2017_unsw_nb15": "CICIDS2017 + UNSW-NB15",
    "cicids2017_iot23": "CICIDS2017 + IoT-23",
    "unsw_nb15_iot23": "UNSW-NB15 + IoT-23",
}
SOURCE_ABBREVIATIONS = {
    "cicids2017": "CIC",
    "unsw_nb15": "UNSW",
    "iot23": "IoT",
    "cicids2017_unsw_nb15": "CIC+UNSW",
    "cicids2017_iot23": "CIC+IoT",
    "unsw_nb15_iot23": "UNSW+IoT",
}
MODEL_COLORS = {
    "Árvore de decisão": "#4C78A8",
    "Random Forest": "#F58518",
    "XGBoost": "#54A24B",
    "SVM linear": "#E45756",
    "MLP": "#B279A2",
}

EVALUATION_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "precision_benign",
    "recall_benign",
    "f1_benign",
    "f1_macro",
]
CV_METRICS = EVALUATION_METRICS.copy()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

saved_tables = []
saved_figures = []


def save_table(dataframe, filename):
    output_path = TABLES_DIR / filename
    dataframe.to_csv(output_path, index=False)
    saved_tables.append(output_path)
    return output_path


def format_table_for_display(dataframe):
    output = dataframe.copy()
    label_maps = {
        "model": MODEL_LABELS,
        "scenario_type": SCENARIO_LABELS,
        "training_source": SOURCE_LABELS,
        "test_source": SOURCE_LABELS,
    }
    for column, label_map in label_maps.items():
        if column in output.columns:
            output[column] = output[column].map(label_map).fillna(output[column])
    return output


def finalize_figure(
    figure,
    filename,
    width,
    height,
    top_margin=130,
    legend_y=1.08,
):
    figure.update_layout(
        template="plotly_white",
        font=dict(family="Arial", size=13),
        title_x=0.5,
        margin=dict(l=80, r=40, t=top_margin, b=80),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=legend_y,
            xanchor="center",
            x=0.5,
        ),
    )

    svg_path = SVG_DIR / f"{filename}.svg"
    png_path = PNG_DIR / f"{filename}.png"

    display(figure)
    figure.write_image(svg_path, format="svg", width=width, height=height)
    figure.write_image(
        png_path,
        format="png",
        width=width,
        height=height,
        scale=2,
    )

    saved_figures.extend([svg_path, png_path])
    return svg_path, png_path

Chrome usado pelo Kaleido: C:\Users\valde\AppData\Local\plotly\choreographer\deps\chrome-win64\chrome.exe


In [2]:
# carregamento e validação dos resultados
training_summary_files = [
    TRAINING_RESULTS_DIR / model_name / "summary.csv"
    for model_name in MODEL_ORDER
]
training_summary = pd.concat(
    [pd.read_csv(summary_path) for summary_path in training_summary_files],
    ignore_index=True,
)
evaluation = pd.read_csv(EVALUATION_RESULTS_DIR / "evaluation_summary.csv")

assert len(training_summary) == 30, (
    "Os resumos do RandomizedSearchCV devem totalizar 30 registros."
)
assert len(evaluation) == 60, (
    "A avaliação deve possuir 60 registros."
)
assert not training_summary.duplicated(
    ["model", "training_source"]
).any()
assert not evaluation.duplicated(
    ["model", "training_source", "test_source"]
).any()
assert set(evaluation["selection_method"]) == {"randomized_search"}

required_evaluation_metrics = EVALUATION_METRICS + ["f1_weighted"]
required_cv_columns = [
    f"{metric}_{suffix}"
    for metric in CV_METRICS
    for suffix in ["mean", "std"]
]

assert not evaluation[required_evaluation_metrics].isna().any().any(), (
    "Há métricas finais ausentes."
)
assert not training_summary[required_cv_columns].isna().any().any(), (
    "Há métricas de validação cruzada ausentes."
)
for metric in required_evaluation_metrics:
    assert evaluation[metric].between(0, 1).all(), (
        f"Valores inválidos em {metric}."
    )

scenario_counts = (
    evaluation.groupby("scenario_type", as_index=False)
    .size()
    .rename(columns={"size": "evaluations"})
)
expected_scenario_counts = {
    "intra_dataset": 15,
    "cross_dataset": 30,
    "combined_to_unseen": 15,
}
assert scenario_counts.set_index("scenario_type")["evaluations"].to_dict() == (
    expected_scenario_counts
)

validation_summary = pd.DataFrame({
    "item": [
        "resumos de treinamento",
        "avaliações",
        "duplicidades",
        "métricas ausentes",
    ],
    "valor": [
        len(training_summary),
        len(evaluation),
        0,
        0,
    ],
    "status": "ok",
})
display(validation_summary)
display(format_table_for_display(scenario_counts))

,item,valor,status
0,resumos de treinamento,30,ok
1,avaliações,60,ok
2,duplicidades,0,ok
3,métricas ausentes,0,ok


,scenario_type,evaluations
0,Combinado,15
1,Cross-dataset,30
2,Intra-dataset,15


In [3]:
# tabelas de validação cruzada e avaliação
def format_mean_std(mean_value, std_value):
    return f"{mean_value:.4f} ± {std_value:.4f}"


model_order_map = {
    model: index for index, model in enumerate(MODEL_ORDER)
}
training_source_order_map = {
    source: index for index, source in enumerate(TRAINING_SOURCE_ORDER)
}
test_source_order_map = {
    source: index for index, source in enumerate(TEST_SOURCE_ORDER)
}

cv_source = training_summary.copy()
cv_source["model_order"] = cv_source["model"].map(model_order_map)
cv_source["training_source_order"] = (
    cv_source["training_source"].map(training_source_order_map)
)
cv_source = cv_source.sort_values(
    ["training_source_order", "model_order"]
)

cv_presentation = pd.DataFrame({
    "Conjunto de treinamento": cv_source["training_source"].map(SOURCE_LABELS),
    "Modelo": cv_source["model"].map(MODEL_ABBREVIATIONS),
    "Hiperparâmetros vencedores": cv_source["params"],
    "F1 macro (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["f1_macro_mean"],
            cv_source["f1_macro_std"],
        )
    ],
    "Acurácia balanceada (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["balanced_accuracy_mean"],
            cv_source["balanced_accuracy_std"],
        )
    ],
    "Precisão ATTACK (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["precision_attack_mean"],
            cv_source["precision_attack_std"],
        )
    ],
    "Recall ATTACK (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["recall_attack_mean"],
            cv_source["recall_attack_std"],
        )
    ],
    "F1 ATTACK (média ± desvio-padrão)": [
        format_mean_std(mean_value, std_value)
        for mean_value, std_value in zip(
            cv_source["f1_attack_mean"],
            cv_source["f1_attack_std"],
        )
    ],
    "Tempo total de treinamento (s)": cv_source["elapsed_seconds"].round(3),
})
save_table(cv_presentation, "cross_validation_summary.csv")

evaluation_source = evaluation.copy()
evaluation_source["model_order"] = (
    evaluation_source["model"].map(model_order_map)
)
evaluation_source["training_source_order"] = (
    evaluation_source["training_source"].map(training_source_order_map)
)
evaluation_source["test_source_order"] = (
    evaluation_source["test_source"].map(test_source_order_map)
)
evaluation_source = evaluation_source.sort_values([
    "training_source_order",
    "test_source_order",
    "model_order",
])

evaluation_presentation = pd.DataFrame({
    "Conjunto de treinamento": (
        evaluation_source["training_source"].map(SOURCE_LABELS)
    ),
    "Conjunto de teste": (
        evaluation_source["test_source"].map(SOURCE_LABELS)
    ),
    "Cenário de avaliação": (
        evaluation_source["scenario_type"].map(SCENARIO_LABELS)
    ),
    "Modelo": evaluation_source["model"].map(MODEL_ABBREVIATIONS),
    "Acurácia": evaluation_source["accuracy"].round(4),
    "Acurácia balanceada": (
        evaluation_source["balanced_accuracy"].round(4)
    ),
    "Precisão ATTACK": evaluation_source["precision_attack"].round(4),
    "Recall ATTACK": evaluation_source["recall_attack"].round(4),
    "F1 ATTACK": evaluation_source["f1_attack"].round(4),
    "Precisão BENIGN": evaluation_source["precision_benign"].round(4),
    "Recall BENIGN": evaluation_source["recall_benign"].round(4),
    "F1 BENIGN": evaluation_source["f1_benign"].round(4),
    "F1 macro": evaluation_source["f1_macro"].round(4),
})
save_table(evaluation_presentation, "evaluation_all.csv")

cv_metrics_for_evaluation = training_summary[[
    "training_source",
    "model",
    "f1_macro_mean",
    "f1_macro_std",
    "balanced_accuracy_mean",
    "balanced_accuracy_std",
]].copy()
tcc_table_source = evaluation_source.merge(
    cv_metrics_for_evaluation,
    on=["training_source", "model"],
    how="left",
    validate="many_to_one",
)
tcc_table_source = tcc_table_source.sort_values([
    "training_source_order",
    "test_source_order",
    "model_order",
]).reset_index(drop=True)

tcc_cv_columns = [
    "f1_macro_mean",
    "f1_macro_std",
    "balanced_accuracy_mean",
    "balanced_accuracy_std",
]
assert len(tcc_table_source) == len(evaluation_source) == 60
assert not tcc_table_source[tcc_cv_columns].isna().any().any(), (
    "A tabela do TCC possui métricas de validação cruzada ausentes."
)

tcc_results_table = pd.DataFrame({
    "Conjunto de treinamento": (
        tcc_table_source["training_source"].map(SOURCE_LABELS)
    ),
    "Conjunto de teste": (
        tcc_table_source["test_source"].map(SOURCE_LABELS)
    ),
    "Modelo": tcc_table_source["model"].map(MODEL_ABBREVIATIONS),
    "F1 macro": tcc_table_source["f1_macro"].round(4),
    "Acurácia balanceada": (
        tcc_table_source["balanced_accuracy"].round(4)
    ),
    "Precisão BENIGN": (
        tcc_table_source["precision_benign"].round(4)
    ),
    "Recall BENIGN": (
        tcc_table_source["recall_benign"].round(4)
    ),
    "Precisão ATTACK": (
        tcc_table_source["precision_attack"].round(4)
    ),
    "Recall ATTACK": (
        tcc_table_source["recall_attack"].round(4)
    ),
    "F1 macro CV (média)": (
        tcc_table_source["f1_macro_mean"].round(4)
    ),
    "F1 macro CV (desvio-padrão)": (
        tcc_table_source["f1_macro_std"].round(4)
    ),
    "Acurácia balanceada CV (média)": (
        tcc_table_source["balanced_accuracy_mean"].round(4)
    ),
    "Acurácia balanceada CV (desvio-padrão)": (
        tcc_table_source["balanced_accuracy_std"].round(4)
    ),
})
assert not tcc_results_table.isna().any().any(), (
    "A tabela do TCC não deve possuir células vazias."
)
save_table(tcc_results_table, "tcc_results_table.csv")

scenario_presentation_tables = {}
for scenario_type in SCENARIO_ORDER:
    scenario_table = evaluation_presentation[
        evaluation_presentation["Cenário de avaliação"].eq(
            SCENARIO_LABELS[scenario_type]
        )
    ].copy()
    scenario_presentation_tables[scenario_type] = scenario_table
    save_table(
        scenario_table,
        f"evaluation_{SCENARIO_FILE_LABELS[scenario_type]}.csv",
    )

best_algorithm_by_experiment = (
    evaluation.sort_values(
        ["f1_macro", "balanced_accuracy", "f1_attack"],
        ascending=[False, False, False],
    )
    .groupby(["training_source", "test_source"], as_index=False)
    .head(1)
    .sort_values(["training_source", "test_source"])
)
save_table(
    format_table_for_display(
        best_algorithm_by_experiment[
            [
                "training_source",
                "test_source",
                "scenario_type",
                "model",
            ] + EVALUATION_METRICS
        ]
    ),
    "best_algorithm_by_experiment.csv",
)

best_per_scenario = (
    evaluation.sort_values(
        ["scenario_type", "f1_macro", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .groupby("scenario_type", as_index=False)
    .head(1)
    .assign(result="Melhor")
)
worst_per_scenario = (
    evaluation.sort_values(
        ["scenario_type", "f1_macro", "balanced_accuracy"],
        ascending=[True, True, True],
    )
    .groupby("scenario_type", as_index=False)
    .head(1)
    .assign(result="Pior")
)
performance_extremes = pd.concat(
    [best_per_scenario, worst_per_scenario],
    ignore_index=True,
)
save_table(
    format_table_for_display(
        performance_extremes[
            [
                "result",
                "scenario_type",
                "model",
                "training_source",
                "test_source",
            ] + EVALUATION_METRICS
        ]
    ),
    "performance_extremes_by_scenario.csv",
)

metric_discrepancies = (
    evaluation.assign(
        accuracy_minus_f1_attack=(
            evaluation["accuracy"] - evaluation["f1_attack"]
        )
    )
    .nlargest(5, "accuracy_minus_f1_attack")
)
save_table(
    format_table_for_display(
        metric_discrepancies[
            [
                "model",
                "training_source",
                "test_source",
                "scenario_type",
                "accuracy",
                "f1_attack",
                "accuracy_minus_f1_attack",
            ]
        ]
    ),
    "performance_metric_discrepancies.csv",
)

display(cv_presentation)
for scenario_type in SCENARIO_ORDER:
    print(f"Avaliação - {SCENARIO_LABELS[scenario_type]}")
    display(scenario_presentation_tables[scenario_type])

print("Tabela consolidada para o TCC")
display(tcc_results_table)

,Conjunto de treinamento,Modelo,Hiperparâmetros vencedores,F1 macro (média ± desvio-padrão),Acurácia balanceada (média ± desvio-padrão),Precisão ATTACK (média ± desvio-padrão),Recall ATTACK (média ± desvio-padrão),F1 ATTACK (média ± desvio-padrão),Tempo total de treinamento (s)
0,CICIDS2017,DT,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_d...",0.9909 ± 0.0005,0.9920 ± 0.0005,0.9824 ± 0.0009,0.9884 ± 0.0008,0.9854 ± 0.0008,26.332
6,CICIDS2017,RF,"{""classifier__bootstrap"": true, ""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": ""balanced"", ""classifier__c...",0.9929 ± 0.0002,0.9959 ± 0.0001,0.9808 ± 0.0009,0.9966 ± 0.0004,0.9887 ± 0.0003,827.533
12,CICIDS2017,XGB,"{""classifier__base_score"": null, ""classifier__booster"": null, ""classifier__callbacks"": null, ""classifier__colsample_...",0.9924 ± 0.0002,0.9946 ± 0.0004,0.9819 ± 0.0004,0.9939 ± 0.0008,0.9878 ± 0.0004,52.846
18,CICIDS2017,SVM,"{""classifier__C"": 46.00306804490304, ""classifier__class_weight"": ""balanced"", ""classifier__dual"": ""auto"", ""classifier...",0.8909 ± 0.0020,0.9357 ± 0.0013,0.7358 ± 0.0042,0.9573 ± 0.0011,0.8321 ± 0.0029,74.970
24,CICIDS2017,MLP,"{""classifier__activation"": ""relu"", ""classifier__alpha"": 1.3096774493777596e-05, ""classifier__batch_size"": ""auto"", ""c...",0.9911 ± 0.0004,0.9938 ± 0.0006,0.9787 ± 0.0010,0.9929 ± 0.0011,0.9858 ± 0.0007,283.931
1,UNSW-NB15,DT,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_d...",0.9787 ± 0.0011,0.9890 ± 0.0006,0.9400 ± 0.0029,0.9938 ± 0.0005,0.9662 ± 0.0017,17.191
7,UNSW-NB15,RF,"{""classifier__bootstrap"": true, ""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criteri...",0.9800 ± 0.0010,0.9911 ± 0.0005,0.9399 ± 0.0031,0.9982 ± 0.0004,0.9682 ± 0.0016,827.461
13,UNSW-NB15,XGB,"{""classifier__base_score"": null, ""classifier__booster"": null, ""classifier__callbacks"": null, ""classifier__colsample_...",0.9800 ± 0.0011,0.9912 ± 0.0005,0.9401 ± 0.0031,0.9983 ± 0.0003,0.9683 ± 0.0017,55.289
19,UNSW-NB15,SVM,"{""classifier__C"": 46.00306804490304, ""classifier__class_weight"": ""balanced"", ""classifier__dual"": ""auto"", ""classifier...",0.9359 ± 0.0013,0.9229 ± 0.0012,0.9352 ± 0.0030,0.8607 ± 0.0021,0.8964 ± 0.0022,63.123
25,UNSW-NB15,MLP,"{""classifier__activation"": ""relu"", ""classifier__alpha"": 0.0001, ""classifier__batch_size"": ""auto"", ""classifier__beta_...",0.9786 ± 0.0013,0.9897 ± 0.0008,0.9377 ± 0.0034,0.9961 ± 0.0009,0.9660 ± 0.0021,343.103


Avaliação - Intra-dataset


,Conjunto de treinamento,Conjunto de teste,Cenário de avaliação,Modelo,Acurácia,Acurácia balanceada,Precisão ATTACK,Recall ATTACK,F1 ATTACK,Precisão BENIGN,Recall BENIGN,F1 BENIGN,F1 macro
0,CICIDS2017,CICIDS2017,Intra-dataset,DT,0.9943,0.9922,0.9828,0.9888,0.9858,0.9972,0.9957,0.9964,0.9911
3,CICIDS2017,CICIDS2017,Intra-dataset,RF,0.9957,0.9964,0.9812,0.9975,0.9893,0.9994,0.9952,0.9973,0.9933
4,CICIDS2017,CICIDS2017,Intra-dataset,XGB,0.9950,0.9944,0.9821,0.9934,0.9877,0.9983,0.9955,0.9969,0.9923
1,CICIDS2017,CICIDS2017,Intra-dataset,SVM,0.9220,0.9350,0.7339,0.9568,0.8307,0.9883,0.9133,0.9493,0.8900
2,CICIDS2017,CICIDS2017,Intra-dataset,MLP,0.9946,0.9942,0.9797,0.9935,0.9865,0.9984,0.9948,0.9966,0.9916
50,UNSW-NB15,UNSW-NB15,Intra-dataset,DT,0.9868,0.9899,0.9422,0.9950,0.9679,0.9987,0.9847,0.9917,0.9798
53,UNSW-NB15,UNSW-NB15,Intra-dataset,RF,0.9874,0.9915,0.9418,0.9985,0.9693,0.9996,0.9846,0.9920,0.9807
54,UNSW-NB15,UNSW-NB15,Intra-dataset,XGB,0.9874,0.9916,0.9417,0.9987,0.9694,0.9997,0.9846,0.9921,0.9807
51,UNSW-NB15,UNSW-NB15,Intra-dataset,SVM,0.9603,0.9226,0.9366,0.8597,0.8965,0.9656,0.9855,0.9754,0.9360
52,UNSW-NB15,UNSW-NB15,Intra-dataset,MLP,0.9867,0.9907,0.9396,0.9974,0.9676,0.9993,0.9840,0.9916,0.9796


Avaliação - Cross-dataset


,Conjunto de treinamento,Conjunto de teste,Cenário de avaliação,Modelo,Acurácia,Acurácia balanceada,Precisão ATTACK,Recall ATTACK,F1 ATTACK,Precisão BENIGN,Recall BENIGN,F1 BENIGN,F1 macro
10,CICIDS2017,UNSW-NB15,Cross-dataset,DT,0.7846,0.4916,0.0402,0.0034,0.0062,0.7973,0.9799,0.8792,0.4427
13,CICIDS2017,UNSW-NB15,Cross-dataset,RF,0.7999,0.5000,0.0000,0.0000,0.0000,0.8000,0.9999,0.8888,0.4444
14,CICIDS2017,UNSW-NB15,Cross-dataset,XGB,0.7999,0.5000,0.0000,0.0000,0.0000,0.8000,0.9999,0.8889,0.4444
11,CICIDS2017,UNSW-NB15,Cross-dataset,SVM,0.7837,0.4906,0.0253,0.0022,0.0040,0.7969,0.9790,0.8787,0.4413
12,CICIDS2017,UNSW-NB15,Cross-dataset,MLP,0.7994,0.4997,0.0714,0.0002,0.0005,0.7999,0.9992,0.8885,0.4445
5,CICIDS2017,IoT-23,Cross-dataset,DT,0.0939,0.0636,0.0037,0.0131,0.0057,0.3162,0.1141,0.1677,0.0867
8,CICIDS2017,IoT-23,Cross-dataset,RF,0.0961,0.0627,0.0020,0.0069,0.0031,0.3229,0.1184,0.1733,0.0882
9,CICIDS2017,IoT-23,Cross-dataset,XGB,0.0943,0.0615,0.0020,0.0070,0.0031,0.3187,0.1161,0.1702,0.0866
6,CICIDS2017,IoT-23,Cross-dataset,SVM,0.1259,0.1516,0.0517,0.1945,0.0817,0.3507,0.1088,0.1660,0.1239
7,CICIDS2017,IoT-23,Cross-dataset,MLP,0.7872,0.4939,0.0686,0.0051,0.0095,0.7980,0.9827,0.8808,0.4451


Avaliação - Combinado


,Conjunto de treinamento,Conjunto de teste,Cenário de avaliação,Modelo,Acurácia,Acurácia balanceada,Precisão ATTACK,Recall ATTACK,F1 ATTACK,Precisão BENIGN,Recall BENIGN,F1 BENIGN,F1 macro
20,CICIDS2017 + UNSW-NB15,IoT-23,Combinado,DT,0.0975,0.0666,0.0043,0.0152,0.0067,0.3241,0.1180,0.1730,0.0899
23,CICIDS2017 + UNSW-NB15,IoT-23,Combinado,RF,0.0208,0.0167,0.0025,0.0098,0.0040,0.0869,0.0236,0.0371,0.0205
24,CICIDS2017 + UNSW-NB15,IoT-23,Combinado,XGB,0.0175,0.0142,0.0022,0.0087,0.0035,0.0738,0.0198,0.0312,0.0173
21,CICIDS2017 + UNSW-NB15,IoT-23,Combinado,SVM,0.1265,0.1530,0.0524,0.1972,0.0828,0.3515,0.1088,0.1661,0.1245
22,CICIDS2017 + UNSW-NB15,IoT-23,Combinado,MLP,0.1709,0.4106,0.1700,0.8100,0.2810,0.1902,0.0112,0.0211,0.1510
15,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,DT,0.8024,0.5168,0.5856,0.0408,0.0763,0.8055,0.9928,0.8894,0.4828
18,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,RF,0.8047,0.5137,0.8467,0.0287,0.0556,0.8044,0.9987,0.8911,0.4733
19,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,XGB,0.8000,0.5002,0.4167,0.0005,0.0010,0.8001,0.9998,0.8889,0.4449
16,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,SVM,0.7943,0.4973,0.0707,0.0023,0.0045,0.7991,0.9923,0.8853,0.4449
17,CICIDS2017 + IoT-23,UNSW-NB15,Combinado,MLP,0.8050,0.5211,0.6774,0.0479,0.0894,0.8068,0.9943,0.8908,0.4901


Tabela consolidada para o TCC


,Conjunto de treinamento,Conjunto de teste,Modelo,F1 macro,Acurácia balanceada,Precisão BENIGN,Recall BENIGN,Precisão ATTACK,Recall ATTACK,F1 macro CV (média),F1 macro CV (desvio-padrão),Acurácia balanceada CV (média),Acurácia balanceada CV (desvio-padrão)
0,CICIDS2017,CICIDS2017,DT,0.9911,0.9922,0.9972,0.9957,0.9828,0.9888,0.9909,0.0005,0.9920,0.0005
1,CICIDS2017,CICIDS2017,RF,0.9933,0.9964,0.9994,0.9952,0.9812,0.9975,0.9929,0.0002,0.9959,0.0001
2,CICIDS2017,CICIDS2017,XGB,0.9923,0.9944,0.9983,0.9955,0.9821,0.9934,0.9924,0.0002,0.9946,0.0004
3,CICIDS2017,CICIDS2017,SVM,0.8900,0.9350,0.9883,0.9133,0.7339,0.9568,0.8909,0.0020,0.9357,0.0013
4,CICIDS2017,CICIDS2017,MLP,0.9916,0.9942,0.9984,0.9948,0.9797,0.9935,0.9911,0.0004,0.9938,0.0006
5,CICIDS2017,UNSW-NB15,DT,0.4427,0.4916,0.7973,0.9799,0.0402,0.0034,0.9909,0.0005,0.9920,0.0005
6,CICIDS2017,UNSW-NB15,RF,0.4444,0.5000,0.8000,0.9999,0.0000,0.0000,0.9929,0.0002,0.9959,0.0001
7,CICIDS2017,UNSW-NB15,XGB,0.4444,0.5000,0.8000,0.9999,0.0000,0.0000,0.9924,0.0002,0.9946,0.0004
8,CICIDS2017,UNSW-NB15,SVM,0.4413,0.4906,0.7969,0.9790,0.0253,0.0022,0.8909,0.0020,0.9357,0.0013
9,CICIDS2017,UNSW-NB15,MLP,0.4445,0.4997,0.7999,0.9992,0.0714,0.0002,0.9911,0.0004,0.9938,0.0006


In [4]:
# desempenho médio por algoritmo e cenário de avaliação
def plot_metric_by_scenario(metric, metric_label, filename):
    plot_data = (
        evaluation.groupby(
            ["scenario_type", "model"],
            as_index=False,
        )
        .agg(
            metric_mean=(metric, "mean"),
            metric_min=(metric, "min"),
            metric_max=(metric, "max"),
            experiments=(metric, "size"),
        )
    )
    plot_data["distance_to_min"] = (
        plot_data["metric_mean"] - plot_data["metric_min"]
    )
    plot_data["distance_to_max"] = (
        plot_data["metric_max"] - plot_data["metric_mean"]
    )
    plot_data["model_label"] = plot_data["model"].map(MODEL_LABELS)
    plot_data["scenario_label"] = (
        plot_data["scenario_type"].map(SCENARIO_LABELS)
    )

    figure = px.bar(
        plot_data,
        x="model_label",
        y="metric_mean",
        facet_col="scenario_label",
        error_y="distance_to_max",
        error_y_minus="distance_to_min",
        category_orders={
            "model_label": [
                MODEL_LABELS[model] for model in MODEL_ORDER
            ],
            "scenario_label": [
                SCENARIO_LABELS[scenario]
                for scenario in SCENARIO_ORDER
            ],
        },
        custom_data=["experiments", "metric_min", "metric_max"],
        labels={
            "model_label": "Algoritmo",
            "metric_mean": metric_label,
        },
        title=(
            f"{metric_label} por algoritmo e cenário de avaliação"
            "<br><sup>Barras: média entre pares treinamento-teste; "
            "intervalos: mínimo-máximo observado</sup>"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=")[-1]
        )
    )
    figure.update_traces(
        marker_color="#4C78A8",
        hovertemplate=(
            "Algoritmo: %{x}<br>Média: %{y:.4f}<br>"
            "Mínimo: %{customdata[1]:.4f}<br>"
            "Máximo: %{customdata[2]:.4f}<br>"
            "Número de avaliações: %{customdata[0]}<extra></extra>"
        ),
        error_y=dict(color="#333333", thickness=1.5, width=5),
    )
    figure.update_yaxes(range=[0, 1.05], matches="y")
    figure.update_xaxes(tickangle=-25)
    finalize_figure(figure, filename, width=1500, height=520)


plot_metric_by_scenario(
    "f1_attack",
    "F1 da classe ATTACK",
    "f1_attack_by_scenario_type",
)
plot_metric_by_scenario(
    "f1_macro",
    "F1 macro",
    "f1_macro_by_scenario_type",
)
plot_metric_by_scenario(
    "balanced_accuracy",
    "Acurácia balanceada",
    "balanced_accuracy_by_scenario_type",
)
plot_metric_by_scenario(
    "precision_attack",
    "Precisão da classe ATTACK",
    "precision_attack_by_scenario_type",
)
plot_metric_by_scenario(
    "recall_attack",
    "Recall da classe ATTACK",
    "recall_attack_by_scenario_type",
)

generalization_pivot = (
    evaluation.groupby(
        ["training_source", "test_source"]
    )["f1_macro"]
    .mean()
    .unstack("test_source")
    .reindex(
        index=TRAINING_SOURCE_ORDER,
        columns=TEST_SOURCE_ORDER,
    )
    .rename(index=SOURCE_LABELS, columns=SOURCE_LABELS)
)
generalization_figure = px.imshow(
    generalization_pivot,
    text_auto=".3f",
    color_continuous_scale="YlGnBu",
    zmin=0,
    zmax=1,
    aspect="auto",
    labels={
        "x": "Conjunto de teste",
        "y": "Origem do treinamento",
        "color": "F1 macro (média)",
    },
    title=(
        "F1 macro médio por origem de treinamento e conjunto de teste"
        "<br><sup>Cada célula representa a média dos cinco algoritmos "
        "na respectiva combinação treinamento-teste</sup>"
    ),
)
finalize_figure(
    generalization_figure,
    "generalization_f1_macro",
    width=900,
    height=720,
)

(WindowsPath('C:/tcc/network-ids-ml-generalization/results/results_analysis/figures/svg/generalization_f1_macro.svg'),
 WindowsPath('C:/tcc/network-ids-ml-generalization/results/results_analysis/figures/png/generalization_f1_macro.png'))

In [5]:
# extremos observados por cenário de avaliação
def plot_metric_extremes(metric, metric_label, filename):
    best_rows = (
        evaluation.sort_values(
            ["scenario_type", metric, "f1_macro"],
            ascending=[True, False, False],
        )
        .groupby("scenario_type", as_index=False)
        .head(1)
        .assign(result="Máximo observado")
    )
    worst_rows = (
        evaluation.sort_values(
            ["scenario_type", metric, "f1_macro"],
            ascending=[True, True, True],
        )
        .groupby("scenario_type", as_index=False)
        .head(1)
        .assign(result="Mínimo observado")
    )
    extremes = pd.concat(
        [best_rows, worst_rows],
        ignore_index=True,
    )
    extremes["scenario_label"] = (
        extremes["scenario_type"].map(SCENARIO_LABELS)
    )
    extremes["model_label"] = extremes["model"].map(MODEL_LABELS)
    extremes["training_source_label"] = (
        extremes["training_source"].map(SOURCE_LABELS)
    )
    extremes["test_source_label"] = (
        extremes["test_source"].map(SOURCE_LABELS)
    )
    extremes["point_label"] = extremes.apply(
        lambda row: (
            f"{MODEL_ABBREVIATIONS[row['model']]} · "
            f"{row[metric]:.3f}"
        ),
        axis=1,
    )

    figure = go.Figure()
    for scenario_type in SCENARIO_ORDER:
        scenario_rows = extremes[
            extremes["scenario_type"].eq(scenario_type)
        ]
        figure.add_trace(go.Scatter(
            x=[
                scenario_rows[metric].min(),
                scenario_rows[metric].max(),
            ],
            y=[
                SCENARIO_LABELS[scenario_type],
                SCENARIO_LABELS[scenario_type],
            ],
            mode="lines",
            line=dict(color="#B0B0B0", width=3),
            hoverinfo="skip",
            showlegend=False,
        ))

    styles = {
        "Máximo observado": "#2E7D32",
        "Mínimo observado": "#C62828",
    }
    for result_label, color in styles.items():
        result_rows = extremes[
            extremes["result"].eq(result_label)
        ]
        custom_data = result_rows[
            [
                "model_label",
                "training_source_label",
                "test_source_label",
                "f1_attack",
                "balanced_accuracy",
                "f1_macro",
            ]
        ].to_numpy()
        figure.add_trace(go.Scatter(
            x=result_rows[metric],
            y=result_rows["scenario_label"],
            mode="markers+text",
            text=result_rows["point_label"],
            textposition=(
                "top center"
                if result_label == "Máximo observado"
                else "bottom center"
            ),
            name=result_label,
            marker=dict(color=color, size=15),
            cliponaxis=False,
            customdata=custom_data,
            hovertemplate=(
                "Cenário: %{y}<br>"
                f"Extremo: {result_label}<br>"
                "Algoritmo: %{customdata[0]}<br>"
                "Treinamento: %{customdata[1]}<br>"
                "Teste: %{customdata[2]}<br>"
                f"{metric_label}: %{{x:.4f}}<br>"
                "F1 ATTACK: %{customdata[3]:.4f}<br>"
                "Acurácia balanceada: %{customdata[4]:.4f}<br>"
                "F1 macro: %{customdata[5]:.4f}<extra></extra>"
            ),
        ))

    figure.update_xaxes(
        title_text=metric_label,
        range=[-0.05, 1.05],
    )
    figure.update_yaxes(
        title_text="Cenário de avaliação",
        categoryorder="array",
        categoryarray=[
            SCENARIO_LABELS[scenario]
            for scenario in reversed(SCENARIO_ORDER)
        ],
    )
    figure.update_layout(
        title=f"Amplitude observada de {metric_label.lower()} por cenário de avaliação"
    )
    finalize_figure(
        figure,
        filename,
        width=1250,
        height=600,
    )


plot_metric_extremes(
    "f1_attack",
    "F1 da classe ATTACK",
    "f1_attack_best_worst_by_scenario",
)
plot_metric_extremes(
    "balanced_accuracy",
    "Acurácia balanceada",
    "balanced_accuracy_best_worst_by_scenario",
)
plot_metric_extremes(
    "f1_macro",
    "F1 macro",
    "f1_macro_best_worst_by_scenario",
)

In [6]:
# comparação entre origens individuais e treinamento combinado
TARGET_TRAINING_SOURCES = {
    "cicids2017": ("unsw_nb15", "iot23", "unsw_nb15_iot23"),
    "unsw_nb15": ("cicids2017", "iot23", "cicids2017_iot23"),
    "iot23": ("cicids2017", "unsw_nb15", "cicids2017_unsw_nb15"),
}

combined_comparison_rows = []
for test_source, sources in TARGET_TRAINING_SOURCES.items():
    individual_source_1, individual_source_2, combined_source = sources

    for model in MODEL_ORDER:
        experiment_rows = evaluation[
            evaluation["test_source"].eq(test_source)
            & evaluation["model"].eq(model)
            & evaluation["training_source"].isin(sources)
        ].set_index("training_source")

        assert len(experiment_rows) == 3, (
            f"Esperados três resultados para {model} "
            f"e teste em {test_source}."
        )

        metric_values = {}
        for metric in ["f1_macro", "balanced_accuracy", "f1_attack"]:
            individual_value_1 = float(
                experiment_rows.loc[individual_source_1, metric]
            )
            individual_value_2 = float(
                experiment_rows.loc[individual_source_2, metric]
            )
            combined_value = float(
                experiment_rows.loc[combined_source, metric]
            )
            metric_values.update({
                f"individual_{metric}_1": individual_value_1,
                f"individual_{metric}_2": individual_value_2,
                f"combined_{metric}": combined_value,
                f"best_individual_{metric}": max(
                    individual_value_1,
                    individual_value_2,
                ),
                f"combined_vs_best_individual_{metric}_delta": (
                    combined_value
                    - max(individual_value_1, individual_value_2)
                ),
            })

        combined_comparison_rows.append({
            "test_source": test_source,
            "model": model,
            "individual_source_1": individual_source_1,
            "individual_source_2": individual_source_2,
            "combined_source": combined_source,
            **metric_values,
        })

combined_comparison = pd.DataFrame(combined_comparison_rows)
assert len(combined_comparison) == 15

combined_comparison_presentation = pd.DataFrame({
    "Conjunto de teste": combined_comparison["test_source"].map(SOURCE_LABELS),
    "Modelo": combined_comparison["model"].map(MODEL_ABBREVIATIONS),
    "Origem individual A": (
        combined_comparison["individual_source_1"].map(SOURCE_LABELS)
    ),
    "F1 macro - A": combined_comparison["individual_f1_macro_1"].round(4),
    "Acurácia balanceada - A": (
        combined_comparison["individual_balanced_accuracy_1"].round(4)
    ),
    "F1 ATTACK - A": combined_comparison["individual_f1_attack_1"].round(4),
    "Origem individual B": (
        combined_comparison["individual_source_2"].map(SOURCE_LABELS)
    ),
    "F1 macro - B": combined_comparison["individual_f1_macro_2"].round(4),
    "Acurácia balanceada - B": (
        combined_comparison["individual_balanced_accuracy_2"].round(4)
    ),
    "F1 ATTACK - B": combined_comparison["individual_f1_attack_2"].round(4),
    "Origens combinadas A + B": (
        combined_comparison["combined_source"].map(SOURCE_LABELS)
    ),
    "F1 macro - A + B": combined_comparison["combined_f1_macro"].round(4),
    "Acurácia balanceada - A + B": (
        combined_comparison["combined_balanced_accuracy"].round(4)
    ),
    "F1 ATTACK - A + B": combined_comparison["combined_f1_attack"].round(4),
    "Δ F1 macro: combinado - melhor individual": (
        combined_comparison[
            "combined_vs_best_individual_f1_macro_delta"
        ].round(4)
    ),
    "Δ acurácia balanceada: combinado - melhor individual": (
        combined_comparison[
            "combined_vs_best_individual_balanced_accuracy_delta"
        ].round(4)
    ),
    "Δ F1 ATTACK: combinado - melhor individual": (
        combined_comparison[
            "combined_vs_best_individual_f1_attack_delta"
        ].round(4)
    ),
})
save_table(
    combined_comparison_presentation,
    "combined_vs_individual_training_sources.csv",
)
display(combined_comparison_presentation)

combined_plot_data = combined_comparison.copy()
combined_plot_data["test_source_label"] = (
    combined_plot_data["test_source"].map(SOURCE_LABELS)
)
combined_plot_data["model_label"] = (
    combined_plot_data["model"].map(MODEL_LABELS)
)


def plot_combined_vs_best_individual(metric, metric_label, filename):
    delta_column = f"combined_vs_best_individual_{metric}_delta"
    plot_data = combined_plot_data.copy()
    plot_data["delta_label"] = plot_data[delta_column].map(
        lambda value: f"{value:+.3f}"
    )

    figure = px.bar(
        plot_data,
        x="model_label",
        y=delta_column,
        facet_col="test_source_label",
        text="delta_label",
        custom_data=["test_source_label"],
        category_orders={
            "model_label": [
                MODEL_LABELS[model] for model in MODEL_ORDER
            ],
            "test_source_label": [
                SOURCE_LABELS[source] for source in TEST_SOURCE_ORDER
            ],
        },
        labels={
            "model_label": "Algoritmo",
            delta_column: f"Δ {metric_label}",
        },
        title=(
            f"Efeito do treinamento combinado sobre {metric_label}"
            "<br><sup>Δ = resultado com duas origens - maior "
            "resultado com uma origem; teste no terceiro conjunto</sup>"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=(
                "Conjunto de teste: "
                f"{annotation.text.split('=')[-1]}"
            )
        )
    )
    figure.update_traces(
        marker_color="#4C78A8",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "Algoritmo: %{x}<br>"
            "Conjunto de teste: %{customdata[0]}<br>"
            f"Δ {metric_label}: %{{y:+.4f}}<extra></extra>"
        ),
    )
    figure.add_hline(y=0, line_width=1, line_color="#444444")
    delta_limit = max(
        float(plot_data[delta_column].abs().max()) * 1.25,
        0.01,
    )
    figure.update_yaxes(
        range=[-delta_limit, delta_limit],
        matches="y",
    )
    figure.update_xaxes(tickangle=-25)
    finalize_figure(figure, filename, width=1500, height=620)


plot_combined_vs_best_individual(
    "f1_macro",
    "F1 macro",
    "combined_vs_best_individual_f1_macro",
)
plot_combined_vs_best_individual(
    "balanced_accuracy",
    "acurácia balanceada",
    "combined_vs_best_individual_balanced_accuracy",
)
plot_combined_vs_best_individual(
    "f1_attack",
    "F1 da classe ATTACK",
    "combined_vs_best_individual_f1_attack",
)

absolute_comparison_rows = []
configuration_specs = [
    (
        "A (individual)",
        "individual_source_1",
        "individual_f1_macro_1",
        "individual_balanced_accuracy_1",
        "individual_f1_attack_1",
    ),
    (
        "B (individual)",
        "individual_source_2",
        "individual_f1_macro_2",
        "individual_balanced_accuracy_2",
        "individual_f1_attack_2",
    ),
    (
        "A + B (combinado)",
        "combined_source",
        "combined_f1_macro",
        "combined_balanced_accuracy",
        "combined_f1_attack",
    ),
]
for _, row in combined_comparison.iterrows():
    for (
        configuration,
        source_column,
        f1_macro_column,
        balanced_accuracy_column,
        f1_attack_column,
    ) in configuration_specs:
        absolute_comparison_rows.append({
            "test_source": row["test_source"],
            "model": row["model"],
            "training_configuration": configuration,
            "training_source": row[source_column],
            "f1_macro": row[f1_macro_column],
            "balanced_accuracy": row[balanced_accuracy_column],
            "f1_attack": row[f1_attack_column],
        })

absolute_comparison = pd.DataFrame(absolute_comparison_rows)
absolute_comparison["test_source_label"] = (
    absolute_comparison["test_source"].map(SOURCE_LABELS)
)
absolute_comparison["model_label"] = (
    absolute_comparison["model"].map(MODEL_LABELS)
)
absolute_comparison["training_source_label"] = (
    absolute_comparison["training_source"].map(SOURCE_LABELS)
)

facet_titles = {
    SOURCE_LABELS[test_source]: (
        f"Conjunto de teste: {SOURCE_LABELS[test_source]}"
        f"<br><sup>A: {SOURCE_LABELS[sources[0]]}; "
        f"B: {SOURCE_LABELS[sources[1]]}</sup>"
    )
    for test_source, sources in TARGET_TRAINING_SOURCES.items()
}
configuration_colors = {
    "A (individual)": "#4C78A8",
    "B (individual)": "#F58518",
    "A + B (combinado)": "#54A24B",
}


def plot_individual_and_combined(metric, metric_label, filename):
    figure = px.bar(
        absolute_comparison,
        x="model_label",
        y=metric,
        color="training_configuration",
        facet_col="test_source_label",
        barmode="group",
        custom_data=["training_source_label"],
        color_discrete_map=configuration_colors,
        category_orders={
            "model_label": [
                MODEL_LABELS[model] for model in MODEL_ORDER
            ],
            "training_configuration": [
                "A (individual)",
                "B (individual)",
                "A + B (combinado)",
            ],
            "test_source_label": [
                SOURCE_LABELS[source] for source in TEST_SOURCE_ORDER
            ],
        },
        labels={
            "model_label": "Algoritmo",
            metric: metric_label,
            "training_configuration": "Composição do treinamento",
        },
        title=(
            f"{metric_label} por composição do conjunto de treinamento"
            "<br><sup>As origens A e B são avaliadas separadamente "
            "e combinadas; o terceiro conjunto é usado para teste</sup>"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=facet_titles[annotation.text.split("=")[-1]]
        )
    )
    figure.update_traces(
        hovertemplate=(
            "Algoritmo: %{x}<br>"
            "Composição do treinamento: %{fullData.name}<br>"
            "Origem(ns) do treinamento: %{customdata[0]}<br>"
            f"{metric_label}: %{{y:.4f}}<extra></extra>"
        )
    )
    figure.update_yaxes(range=[0, 1.05], matches="y")
    figure.update_xaxes(tickangle=-25)
    finalize_figure(figure, filename, width=1600, height=650)


plot_individual_and_combined(
    "f1_macro",
    "F1 macro",
    "individual_and_combined_f1_macro",
)
plot_individual_and_combined(
    "balanced_accuracy",
    "Acurácia balanceada",
    "individual_and_combined_balanced_accuracy",
)
plot_individual_and_combined(
    "f1_attack",
    "F1 da classe ATTACK",
    "individual_and_combined_f1_attack",
)

,Conjunto de teste,Modelo,Origem individual A,F1 macro - A,Acurácia balanceada - A,F1 ATTACK - A,Origem individual B,F1 macro - B,Acurácia balanceada - B,F1 ATTACK - B,Origens combinadas A + B,F1 macro - A + B,Acurácia balanceada - A + B,F1 ATTACK - A + B,Δ F1 macro: combinado - melhor individual,Δ acurácia balanceada: combinado - melhor individual,Δ F1 ATTACK: combinado - melhor individual
0,CICIDS2017,DT,UNSW-NB15,0.4297,0.4249,0.1297,IoT-23,0.7673,0.7483,0.6186,UNSW-NB15 + IoT-23,0.4208,0.4494,0.0073,-0.3465,-0.2989,-0.6113
1,CICIDS2017,RF,UNSW-NB15,0.4241,0.4201,0.1485,IoT-23,0.4517,0.4983,0.0205,UNSW-NB15 + IoT-23,0.4877,0.4929,0.1456,0.0360,-0.0054,-0.0028
2,CICIDS2017,XGB,UNSW-NB15,0.4029,0.3970,0.0690,IoT-23,0.4511,0.4923,0.0273,UNSW-NB15 + IoT-23,0.5043,0.5048,0.1917,0.0532,0.0125,0.1227
3,CICIDS2017,SVM,UNSW-NB15,0.6994,0.7015,0.5207,IoT-23,0.4850,0.4853,0.1987,UNSW-NB15 + IoT-23,0.4954,0.6804,0.4398,-0.2040,-0.0211,-0.0809
4,CICIDS2017,MLP,UNSW-NB15,0.6319,0.6659,0.4514,IoT-23,0.4602,0.4964,0.0459,UNSW-NB15 + IoT-23,0.4410,0.4466,0.0838,-0.1909,-0.2193,-0.3676
5,UNSW-NB15,DT,CICIDS2017,0.4427,0.4916,0.0062,IoT-23,0.5549,0.5538,0.2826,CICIDS2017 + IoT-23,0.4828,0.5168,0.0763,-0.0720,-0.0370,-0.2063
6,UNSW-NB15,RF,CICIDS2017,0.4444,0.5000,0.0000,IoT-23,0.5061,0.5279,0.1230,CICIDS2017 + IoT-23,0.4733,0.5137,0.0556,-0.0328,-0.0142,-0.0674
7,UNSW-NB15,XGB,CICIDS2017,0.4444,0.5000,0.0000,IoT-23,0.3424,0.3233,0.0724,CICIDS2017 + IoT-23,0.4449,0.5002,0.0010,0.0005,0.0002,-0.0714
8,UNSW-NB15,SVM,CICIDS2017,0.4413,0.4906,0.0040,IoT-23,0.4859,0.5079,0.1003,CICIDS2017 + IoT-23,0.4449,0.4973,0.0045,-0.0410,-0.0106,-0.0958
9,UNSW-NB15,MLP,CICIDS2017,0.4445,0.4997,0.0005,IoT-23,0.4543,0.4567,0.1142,CICIDS2017 + IoT-23,0.4901,0.5211,0.0894,0.0359,0.0214,-0.0248


In [7]:
# distribuição das 60 avaliações e resumo dos artefatos
plot_data = evaluation.copy()
plot_data["model_label"] = plot_data["model"].map(MODEL_LABELS)
plot_data["scenario_label"] = (
    plot_data["scenario_type"].map(SCENARIO_LABELS)
)
plot_data["training_source_label"] = (
    plot_data["training_source"].map(SOURCE_LABELS)
)
plot_data["test_source_label"] = (
    plot_data["test_source"].map(SOURCE_LABELS)
)

absolute_evaluation_metrics = {
    "f1_attack": (
        "F1 da classe ATTACK",
        "f1_attack_by_algorithm_and_scenario",
    ),
    "balanced_accuracy": (
        "Acurácia balanceada",
        "balanced_accuracy_by_algorithm_and_scenario",
    ),
    "f1_macro": (
        "F1 macro",
        "f1_macro_by_algorithm_and_scenario",
    ),
}

for metric, (metric_label, filename) in (
    absolute_evaluation_metrics.items()
):
    figure = px.scatter(
        plot_data,
        x=metric,
        y="model_label",
        color="model_label",
        facet_col="scenario_label",
        color_discrete_map=MODEL_COLORS,
        category_orders={
            "model_label": [
                MODEL_LABELS[model] for model in MODEL_ORDER
            ],
            "scenario_label": [
                SCENARIO_LABELS[scenario]
                for scenario in SCENARIO_ORDER
            ],
        },
        custom_data=[
            "training_source_label",
            "test_source_label",
        ],
        labels={
            metric: metric_label,
            "model_label": "Algoritmo",
        },
        title=(
            f"{metric_label} por algoritmo e cenário de avaliação"
            "<br><sup>Cada ponto representa uma combinação de "
            "origem de treinamento, conjunto de teste e algoritmo</sup>"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=")[-1]
        )
    )
    figure.update_traces(
        marker=dict(size=9, opacity=0.75),
        hovertemplate=(
            "Algoritmo: %{y}<br>"
            "Treinamento: %{customdata[0]}<br>"
            "Teste: %{customdata[1]}<br>"
            f"{metric_label}: %{{x:.4f}}<extra></extra>"
        ),
    )
    figure.update_xaxes(range=[0, 1.02], matches="x")
    figure.update_layout(showlegend=False)
    finalize_figure(
        figure,
        filename,
        width=1500,
        height=650,
    )

evaluation_routes = (
    plot_data["training_source"].map(SOURCE_ABBREVIATIONS)
    + " → "
    + plot_data["test_source"].map(SOURCE_ABBREVIATIONS)
)
plot_data["evaluation_route"] = evaluation_routes
EVALUATION_ROUTES_BY_SCENARIO = {
    "intra_dataset": [
        "CIC → CIC",
        "UNSW → UNSW",
        "IoT → IoT",
    ],
    "cross_dataset": [
        "CIC → UNSW",
        "CIC → IoT",
        "UNSW → CIC",
        "UNSW → IoT",
        "IoT → CIC",
        "IoT → UNSW",
    ],
    "combined_to_unseen": [
        "CIC+UNSW → IoT",
        "CIC+IoT → UNSW",
        "UNSW+IoT → CIC",
    ],
}


def plot_all_training_test_combinations(
    metric,
    metric_label,
    filename,
):
    figure = px.strip(
        plot_data,
        x="evaluation_route",
        y=metric,
        color="model_label",
        facet_col="scenario_label",
        stripmode="group",
        custom_data=[
            "model_label",
            "training_source_label",
            "test_source_label",
            "scenario_label",
        ],
        color_discrete_map=MODEL_COLORS,
        category_orders={
            "model_label": [
                MODEL_LABELS[model] for model in MODEL_ORDER
            ],
            "scenario_label": [
                SCENARIO_LABELS[scenario]
                for scenario in SCENARIO_ORDER
            ],
        },
        labels={
            "evaluation_route": "",
            metric: metric_label,
            "model_label": "Algoritmo",
        },
        title=(
            f"Resultados individuais de {metric_label} por "
            "combinação treinamento-teste"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=(
                "Cenário: "
                f"{annotation.text.split('=')[-1]}"
            )
        )
    )
    figure.update_layout(
        title=dict(
            y=0.985,
            yanchor="top",
        ),
    )
    for axis_name, scenario_type in zip(
        ["xaxis", "xaxis2", "xaxis3"],
        SCENARIO_ORDER,
    ):
        figure.layout[axis_name].update(
            categoryorder="array",
            categoryarray=(
                EVALUATION_ROUTES_BY_SCENARIO[scenario_type]
            ),
        )
    for column_index, scenario_type in enumerate(
        SCENARIO_ORDER,
        start=1,
    ):
        route_count = len(
            EVALUATION_ROUTES_BY_SCENARIO[scenario_type]
        )
        for separator_index in range(1, route_count):
            figure.add_vline(
                x=separator_index - 0.5,
                row=1,
                col=column_index,
                line_width=1,
                line_dash="dash",
                line_color="rgba(80, 80, 80, 0.28)",
            )
    figure.update_traces(
        jitter=0.12,
        marker=dict(size=9, opacity=0.82),
        hovertemplate=(
            "Algoritmo: %{customdata[0]}<br>"
            "Origem do treinamento: %{customdata[1]}<br>"
            "Conjunto de teste: %{customdata[2]}<br>"
            "Cenário: %{customdata[3]}<br>"
            f"{metric_label}: %{{y:.4f}}<extra></extra>"
        ),
    )
    figure.update_xaxes(
        matches=None,
        tickangle=-30,
        title_text=None,
        showgrid=False,
    )
    figure.update_yaxes(
        range=[0, 1.02],
        matches="y",
        showticklabels=True,
    )
    finalize_figure(
        figure,
        filename,
        width=1800,
        height=580,
        top_margin=115,
        legend_y=1.10,
    )


plot_all_training_test_combinations(
    "f1_macro",
    "F1 macro",
    "f1_macro_all_training_test_combinations",
)
plot_all_training_test_combinations(
    "balanced_accuracy",
    "Acurácia balanceada",
    "balanced_accuracy_all_training_test_combinations",
)

generated_tables = sorted(
    path.relative_to(TABLES_DIR).as_posix()
    for path in saved_tables
)
generated_svg = sorted(
    path.name for path in SVG_DIR.glob("*.svg")
)
generated_png = sorted(
    path.name for path in PNG_DIR.glob("*.png")
)

assert len(generated_svg) == len(generated_png)
assert len(generated_svg) == 20, (
    "A análise deve gerar exatamente 20 gráficos."
)
assert {
    Path(filename).stem for filename in generated_svg
} == {
    Path(filename).stem for filename in generated_png
}

print(f"Tabelas salvas: {len(generated_tables)}")
print(f"Gráficos salvos em SVG: {len(generated_svg)}")
print(f"Gráficos salvos em PNG: {len(generated_png)}")
display(pd.DataFrame({"tabelas": generated_tables}))
display(pd.DataFrame({
    "grafico_svg": generated_svg,
    "grafico_png": generated_png,
}))

Tabelas salvas: 10
Gráficos salvos em SVG: 20
Gráficos salvos em PNG: 20


,tabelas
0,best_algorithm_by_experiment.csv
1,combined_vs_individual_training_sources.csv
2,cross_validation_summary.csv
3,evaluation_all.csv
4,evaluation_combined.csv
5,evaluation_cross_dataset.csv
6,evaluation_intra_dataset.csv
7,performance_extremes_by_scenario.csv
8,performance_metric_discrepancies.csv
9,tcc_results_table.csv


,grafico_svg,grafico_png
0,balanced_accuracy_all_training_test_combinations.svg,balanced_accuracy_all_training_test_combinations.png
1,balanced_accuracy_best_worst_by_scenario.svg,balanced_accuracy_best_worst_by_scenario.png
2,balanced_accuracy_by_algorithm_and_scenario.svg,balanced_accuracy_by_algorithm_and_scenario.png
3,balanced_accuracy_by_scenario_type.svg,balanced_accuracy_by_scenario_type.png
4,combined_vs_best_individual_balanced_accuracy.svg,combined_vs_best_individual_balanced_accuracy.png
5,combined_vs_best_individual_f1_attack.svg,combined_vs_best_individual_f1_attack.png
6,combined_vs_best_individual_f1_macro.svg,combined_vs_best_individual_f1_macro.png
7,f1_attack_best_worst_by_scenario.svg,f1_attack_best_worst_by_scenario.png
8,f1_attack_by_algorithm_and_scenario.svg,f1_attack_by_algorithm_and_scenario.png
9,f1_attack_by_scenario_type.svg,f1_attack_by_scenario_type.png
